In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
# Fetch TMI (Thesaurus Musicarum Italicarum) index and extract links
tmi_index_url = "https://tmiweb.science.uu.nl/text/index.html"
tmi_base_url = "https://tmiweb.science.uu.nl/text/"

response = requests.get(tmi_index_url)
soup = BeautifulSoup(response.content, 'html.parser')

# Extract all reading-edition links (excluding the index itself)
tmi_links = [
    tmi_base_url + link['href']
    for link in soup.find_all('a', href=True)
    if link['href'].startswith('reading-edition/') 
    and link['href'] != 'reading-edition/index.html'
]

print(f"Found {len(tmi_links)} TMI text links")
for link in tmi_links[:5]:
    print(f"  {link}")
print("...")

Found 34 TMI text links
  https://tmiweb.science.uu.nl/text/reading-edition/aarcom.html
  https://tmiweb.science.uu.nl/text/reading-edition/aarluc.html
  https://tmiweb.science.uu.nl/text/reading-edition/aartos.html
  https://tmiweb.science.uu.nl/text/reading-edition/aartra.html
  https://tmiweb.science.uu.nl/text/reading-edition/aartrm.html
...


In [3]:
tmi_links

['https://tmiweb.science.uu.nl/text/reading-edition/aarcom.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/aarluc.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/aartos.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/aartra.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/aartrm.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/agadel.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/agamus.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/aigill.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/aigtes.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/artart.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/artcon86.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/artcon89.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/artdis.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/artimp.html',
 'https://tmiweb.science.uu.nl/text/reading-edition/artsec.html',
 'http

In [4]:
import os
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Create output folder if it doesn't exist
output_folder = 'tmi_sources'
os.makedirs(output_folder, exist_ok=True)

# Set up a session with retry logic
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)

# Fetch and save each URL
for i, url in enumerate(tmi_links):
    filename = url.split('/')[-1]
    filepath = os.path.join(output_folder, filename)
    
    if os.path.exists(filepath):
        print(f"Skipping {filename} (already exists)")
        continue
    
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(response.text)
        
        print(f"[{i+1}/{len(tmi_links)}] Saved: {filename}")
        
        # Delay to be polite to the server
        time.sleep(1)
        
    except requests.RequestException as e:
        print(f"Error fetching {filename}: {e}")

print(f"\nDone! Files saved to '{output_folder}/' folder")

[1/34] Saved: aarcom.html
[2/34] Saved: aarluc.html
[3/34] Saved: aartos.html
[4/34] Saved: aartra.html
[5/34] Saved: aartrm.html
[6/34] Saved: agadel.html
[7/34] Saved: agamus.html
[8/34] Saved: aigill.html
[9/34] Saved: aigtes.html
[10/34] Saved: artart.html
[11/34] Saved: artcon86.html
[12/34] Saved: artcon89.html
[13/34] Saved: artdis.html
[14/34] Saved: artimp.html
[15/34] Saved: artsec.html
[16/34] Saved: balcro.html
[17/34] Saved: balvit.html
[18/34] Saved: bonreg.html
[19/34] Saved: botdes.html
[20/34] Saved: dlabre.html
[21/34] Saved: fenreg.html
[22/34] Saved: galdia.html
[23/34] Saved: galdis.html
[24/34] Saved: meidis.html
[25/34] Saved: pondia.html
[26/34] Saved: rodreg.html
[27/34] Saved: tigcom.html
[28/34] Saved: vecmos.html
[29/34] Saved: vicant.html
[30/34] Saved: zacpra1.html
[31/34] Saved: zardim89.html
[32/34] Saved: zarins58.html
[33/34] Saved: zarins89.html
[34/34] Saved: zarsop.html

Done! Files saved to 'tmi_sources/' folder
